# Feature Engineering

The exploratory analysis revealed several features that are strongly associated with house prices, as well as groups of variables that describe related aspects of a property. In this notebook, we leverage this information to create new features that may help machine learning models capture important patterns in the data.

Feature engineering is the process of transforming existing variables into new representations that better reflect the underlying factors influencing the target variable. By combining related features, aggregating information, and creating meaningful indicators, we can often provide models with a more informative view of the problem than the raw data alone.

The features introduced in this notebook are motivated by simple domain knowledge about residential properties. In general, buyers tend to care about factors such as total living space, property age, overall quality, available amenities, and the amount of usable space dedicated to specific purposes. The engineered features below aim to capture these concepts more directly.

Before creating new features, we first load the processed dataset generated in the previous notebook. This dataset already includes the missing-value handling and data quality corrections identified during the exploratory data analysis stage.

We then import the libraries required for feature engineering and subsequent preprocessing steps.

In [30]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

sns.set_theme()

# Load data processed and saved in the EDA notebook
df_full = pd.read_parquet(
    "processed_data/01_data.parquet"
)
df_features = pd.read_parquet(
    "processed_data/01_features.parquet"
)

In [31]:
df_full

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,IsTrainSet
0,1,60,RL,65.0,8450,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,2,2008,WD,Normal,208500.0,True
1,2,20,RL,80.0,9600,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,5,2007,WD,Normal,181500.0,True
2,3,60,RL,68.0,11250,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,9,2008,WD,Normal,223500.0,True
3,4,70,RL,60.0,9550,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,2,2006,WD,Abnorml,140000.0,True
4,5,60,RL,84.0,14260,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,12,2008,WD,Normal,250000.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2915,160,RM,21.0,1936,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,6,2006,WD,Normal,NaN,False
2915,2916,160,RM,21.0,1894,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,4,2006,WD,Abnorml,NaN,False
2916,2917,20,RL,160.0,20000,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,9,2006,WD,Abnorml,NaN,False
2917,2918,85,RL,62.0,10441,Pave,None,Reg,Lvl,AllPub,...,None,MnPrv,Shed,700,7,2006,WD,Normal,NaN,False


In [32]:
df_features

,Name,Type,Category,Source,Description
0,MSSubClass,numerical,original,MSSubClass,Identifies the type of dwelling involved in th...
1,MSZoning,categorical,original,MSZoning,Identifies the general zoning classification o...
2,LotFrontage,numerical,original,LotFrontage,Linear feet of street connected to property
3,LotArea,numerical,original,LotArea,Lot size in square feet
4,Street,categorical,original,Street,Type of road access to property
...,...,...,...,...,...
74,MiscVal,numerical,original,MiscVal,$Value of miscellaneous feature
75,MoSold,numerical,original,MoSold,Month Sold (MM)
76,YrSold,numerical,original,YrSold,Year Sold (YYYY)
77,SaleType,categorical,original,SaleType,Type of sale


## Engineered aggregate features

Aggregate features combine information from multiple variables into a single measure. The goal is to capture broader property characteristics that may be more informative than the individual components alone. For example, total living space or total bathroom count may provide a more complete description of a property than considering each contributing feature separately.

### TotalRooms

The total number of rooms is often more informative than the individual room counts alone. This feature attempts to capture the overall functional capacity of the house.

In [33]:
# Create the feature
df_full["TotalRooms"] = (
    df_full["TotRmsAbvGrd"]
    + df_full["KitchenAbvGr"]
)

# Add feature to df_features
new_row = {
    "Name": "TotalRooms",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "TotRmsAbvGrd;KitchenAbvGr",
    "Description": "Total number of living spaces and kitchens"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### TotalBathrooms

The dataset distributes bathroom information across multiple variables. Combining them into a single measure provides a more complete representation of the property's sanitary facilities.

In [34]:
# Create the feature
df_full["TotalBathrooms"] = (
    df_full["FullBath"]
    + 0.5*df_full["HalfBath"]
    + df_full["BsmtFullBath"]
    + 0.5*df_full["BsmtHalfBath"]
)

# Add feature to df_features
new_row = {
    "Name": "TotalBathrooms",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "FullBath;HalfBath;BsmtFullBath;BsmtHalfBath",
    "Description": "Total number of bathrooms, accounting for half bathrooms"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### TotalArea

The original dataset stores living space across multiple variables corresponding to different levels of the house. While this information is useful, buyers are often interested in the total amount of usable space available. This feature aggregates the main living areas into a single measure of property size.

In [35]:
# Create the feature
df_full["TotalArea"] = (
    df_full["TotalBsmtSF"]
    + df_full["1stFlrSF"]
    + df_full["2ndFlrSF"]
)

# Add feature to df_features
new_row = {
    "Name": "TotalArea",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "TotalBsmtSF;1stFlrSF;2ndFlrSF",
    "Description": "Combined usable area of the house"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### TotalPorchArea

Porches appear in several separate variables depending on their type. This feature aggregates them into a single measure of outdoor living space.

In [36]:
# Create the feature
df_full["TotalPorchArea"] = (
    df_full["OpenPorchSF"]
    + df_full["EnclosedPorch"]
    + df_full["3SsnPorch"]
    + df_full["ScreenPorch"]
)

# Add feature to df_features
new_row = {
    "Name": "TotalPorchArea",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "OpenPorchSF;EnclosedPorch;3SsnPorch;ScreenPorch",
    "Description": "Combined area of all porch spaces"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### TotalQualityScore

The overall quality and overall condition ratings describe complementary aspects of a property. Combining them creates a broader measure of the house's general quality.

In [37]:
# Create the feature
df_full["TotalQualityScore"] = (
    df_full["OverallQual"]
    + df_full["OverallCond"]
)

# Add feature to df_features
new_row = {
    "Name": "TotalQualityScore",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "OverallQual;OverallCond",
    "Description": "Combined measure of overall quality and condition"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### LuxuryFeatureCount

This feature counts the number of selected amenities present in a property. While each amenity contributes information individually, combining them into a single feature provides a simple measure of the property's amenity level.

The term luxury is used loosely here to refer to features that are commonly associated with larger or more valuable homes. Higher values indicate that a property offers a greater number of these amenities and may therefore command a higher market price.

In [38]:
# Create the feature
df_full["LuxuryFeatureCount"] = (
    (df_full["PoolArea"] > 0).astype(int)
    + (df_full["GarageArea"] > 0).astype(int)
    + (df_full["TotalBsmtSF"] > 0).astype(int)
    + (df_full["Fireplaces"] > 0).astype(int)
)

# Add feature to df_features
new_row = {
    "Name": "LuxuryFeatureCount",
    "Type": "numerical",
    "Category": "aggregate",
    "Source": "HasPool;HasGarage;HasBasement;HasFireplace",
    "Description": "Count of selected premium property amenities"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

## Engineered difference features

Difference features represent the elapsed time between two related events or measurements. In real estate data, the age of a house, garage, or renovation can be more informative than the corresponding years themselves, as buyers often respond to how old or recently updated a property is rather than the calendar dates involved.


### HouseAge

The age of a property at the time of sale is likely to influence its market value. Newer houses often command higher prices due to modern construction standards, lower maintenance requirements, and updated amenities.

In [39]:
# Create the feature
df_full["HouseAge"] = (
    df_full["YrSold"]
    - df_full["YearBuilt"]
)

# Add feature to df_features
new_row = {
    "Name": "HouseAge",
    "Type": "numerical",
    "Category": "difference",
    "Source": "YrSold;YearBuilt",
    "Description": "Age of the property at the time of sale"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### YearsSinceRemodel

The original remodeling year provides useful information, but the number of years since the last renovation may be more directly related to a property's perceived condition and market value.

In [40]:
# Create the feature
df_full["YearsSinceRemodel"] = (
    df_full["YrSold"]
    - df_full["YearRemodAdd"]
)

# Add feature to df_features
new_row = {
    "Name": "YearsSinceRemodel",
    "Type": "numerical",
    "Category": "difference",
    "Source": "YrSold;YearRemodAdd",
    "Description": "Years elapsed since the most recent remodeling"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### GarageAge

A newer garage may be more desirable than an older one due to reduced wear and improved construction standards. This feature captures the age of the garage at the time of sale.

In [41]:
# Create the feature
df_full["GarageAge"] = (
    df_full["YrSold"]
    - df_full["GarageYrBlt"]
)

# Add feature to df_features
new_row = {
    "Name": "GarageAge",
    "Type": "numerical",
    "Category": "difference",
    "Source": "YrSold;GarageYrBlt",
    "Description": "Age of the garage at the time of sale"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

## Engineered ineteraction features

Interaction features combine two or more variables in a way that allows their joint effect to be represented explicitly. These features can help capture relationships that may not be apparent when considering each variable independently. For example, a large house may be valuable, and a high-quality house may be valuable, but a house that is both large and high quality may command an even greater premium.


### QualityArea

Previous analysis showed that both living area and overall quality are strongly associated with sale price. This feature captures the interaction between these variables, allowing the model to distinguish between houses that are large, high-quality, or both. It can be interpreted as a rough measure of quality-adjusted living space.

In [42]:
# Create the feature
df_full["QualityArea"] = (
    df_full["OverallQual"]
    * df_full["GrLivArea"]
)

# Add feature to df_features
new_row = {
    "Name": "QualityArea",
    "Type": "numerical",
    "Category": "interaction",
    "Source": "OverallQual;GrLivArea",
    "Description": "Interaction between house quality and living area"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

## Engineered ratio features

Ratio features describe the relationship between two quantities rather than their absolute values. These features can provide a more meaningful representation of certain property characteristics, such as the amount of living space per room or the garage area available per parking space.

### GarageAreaPerCar

The total garage area and the number of vehicle spaces provide complementary information about the garage. By dividing the area by the number of parking spaces, we obtain a rough measure of the amount of space available per vehicle.

This feature serves as an example of a ratio-based transformation, which can sometimes reveal relationships that are not apparent from the original variables alone. Larger values may indicate more spacious garages with additional storage or workshop space.

When constructing this feature, care must be taken to avoid division by zero. Properties without a garage have `GarageCars = 0`, so the ratio is defined as 0 for these observations.

In [43]:
# Create the feature
df_full["GarageAreaPerCar"] = np.where(
    df_full["GarageCars"] > 0,
    df_full["GarageArea"] / df_full["GarageCars"],
    0
)

# Add feature to df_features
new_row = {
    "Name": "GarageAreaPerCar",
    "Type": "numerical",
    "Category": "ratio",
    "Source": "GarageArea;GarageCars",
    "Description": "Average garage area available per parking space"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### LivingAreaPerRoom

The total above-ground living area and the number of rooms both provide useful information about the size of a house. However, they describe different aspects of the property. Two houses may have the same living area but very different room layouts.

By dividing the living area by the number of rooms, we obtain a rough measure of the average room size. This ratio may help distinguish between houses with many small rooms and those with fewer, more spacious rooms.

As with `GarageAreaPerCar`, or any ratio-based feature, care must be taken to avoid division by zero. In this dataset, all properties have at least one room above ground, but we nevertheless define the ratio as 0 whenever TotRmsAbvGrd is zero for robustness.

In [44]:
# Create the feature
df_full["LivingAreaPerRoom"] = np.where(
    df_full["TotRmsAbvGrd"] > 0,
    df_full["GrLivArea"] / df_full["TotRmsAbvGrd"],
    0
)

# Add feature to df_features
new_row = {
    "Name": "LivingAreaPerRoom",
    "Type": "numerical",
    "Category": "ratio",
    "Source": "GrLivArea;TotRmsAbvGrd",
    "Description": "Average living area per room"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

## Engineered binary indicator features

Several variables naturally describe the presence or absence of a property characteristic. While these variables can be represented as boolean values, converting them to integers (`0` or `1`) provides a format that is directly compatible with most machine learning algorithms and simplifies subsequent preprocessing.

### IsRemodeled

Indicates whether the property has been remodeled since its original construction.

In [45]:
# Create the feature
df_full["IsRemodeled"] = (
    df_full["YearBuilt"] != df_full["YearRemodAdd"]
).astype(int)

# Add feature to df_features
new_row = {
    "Name": "IsRemodeled",
    "Type": "numerical",
    "Category": "indicator",
    "Source": "YearBuilt;YearRemodAdd",
    "Description": "Indicates whether the property has been remodeled"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### HasPool

Indicates whether the property includes a swimming pool.

In [46]:
# Create the feature
df_full["HasPool"] = (
    df_full["PoolArea"] > 0
).astype(int)

# Add feature to df_features
new_row = {
    "Name": "HasPool",
    "Type": "numerical",
    "Category": "indicator",
    "Source": "PoolArea",
    "Description": "Indicates whether the property has a pool"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### HasGarage

Indicates whether the property includes a garage.

In [47]:
# Create the feature
df_full["HasGarage"] = (
    df_full["GarageArea"] > 0
).astype(int)

# Add feature to df_features
new_row = {
    "Name": "HasGarage",
    "Type": "numerical",
    "Category": "indicator",
    "Source": "YearBuilt;YearRemodAdd",
    "Description": "Indicates whether the property has a garage"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### HasBasement

Indicates whether the property includes a basement.

In [48]:
# Create the feature
df_full["HasBasement"] = (
    df_full["TotalBsmtSF"] > 0
).astype(int)

# Add feature to df_features
new_row = {
    "Name": "HasBasement",
    "Type": "numerical",
    "Category": "indicator",
    "Source": "TotalBsmtSF",
    "Description": "Indicates whether the property has a basement"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

### HasFireplace

Indicates whether the property includes at least one fireplace.

In [49]:
# Create the feature
df_full["HasFireplace"] = (
    df_full["Fireplaces"] > 0
).astype(int)

# Add feature to df_features
new_row = {
    "Name": "HasFireplace",
    "Type": "numerical",
    "Category": "indicator",
    "Source": "YearBuilt;YearRemodAdd",
    "Description": "Indicates whether the property has at least one fireplace"
}
df_features = pd.concat(
    [df_features, pd.DataFrame([new_row])],
    ignore_index=True
)

## Summary of engineered features

Throughout this notebook, we created several new features designed to capture information that is not directly available in the original dataset. These features fall into a number of categories, including aggregate features, time-based features, ratio features, interaction features, and binary indicators.

To facilitate tracking and documentation, information about each feature is stored in the `df_features` metadata table. This table records the type of feature, its category, the source variables used to create it, and a brief description of its meaning.

The following tables summarize the engineered features introduced in this notebook.

### Aggregate features
Aggregate features combine information from multiple variables into a single measure.

In [50]:
df_features.loc[
    df_features["Category"].eq("aggregate")
]

,Name,Type,Category,Source,Description
79,TotalRooms,numerical,aggregate,TotRmsAbvGrd;KitchenAbvGr,Total number of living spaces and kitchens
80,TotalBathrooms,numerical,aggregate,FullBath;HalfBath;BsmtFullBath;BsmtHalfBath,"Total number of bathrooms, accounting for half..."
81,TotalArea,numerical,aggregate,TotalBsmtSF;1stFlrSF;2ndFlrSF,Combined usable area of the house
82,TotalPorchArea,numerical,aggregate,OpenPorchSF;EnclosedPorch;3SsnPorch;ScreenPorch,Combined area of all porch spaces
83,TotalQualityScore,numerical,aggregate,OverallQual;OverallCond,Combined measure of overall quality and condition
84,LuxuryFeatureCount,numerical,aggregate,HasPool;HasGarage;HasBasement;HasFireplace,Count of selected premium property amenities


### Difference features
Difference features capture the elapsed time or distance between two related quantities.

In [51]:
df_features.loc[
    df_features["Category"].eq("difference")
]

,Name,Type,Category,Source,Description
85,HouseAge,numerical,difference,YrSold;YearBuilt,Age of the property at the time of sale
86,YearsSinceRemodel,numerical,difference,YrSold;YearRemodAdd,Years elapsed since the most recent remodeling
87,GarageAge,numerical,difference,YrSold;GarageYrBlt,Age of the garage at the time of sale


### Ratio features
Ratio features describe the relationship between two quantities and often provide a more informative measure than either variable alone.

In [52]:
df_features.loc[
    df_features["Category"].eq("ratio")
]

,Name,Type,Category,Source,Description
89,GarageAreaPerCar,numerical,ratio,GarageArea;GarageCars,Average garage area available per parking space
90,LivingAreaPerRoom,numerical,ratio,GrLivArea;TotRmsAbvGrd,Average living area per room


### Indicator features
Indicator features explicitly represent the presence or absence of important property characteristics.

In [53]:
df_features.loc[
    df_features["Category"].eq("indicator")
]

,Name,Type,Category,Source,Description
91,IsRemodeled,numerical,indicator,YearBuilt;YearRemodAdd,Indicates whether the property has been remodeled
92,HasPool,numerical,indicator,PoolArea,Indicates whether the property has a pool
93,HasGarage,numerical,indicator,YearBuilt;YearRemodAdd,Indicates whether the property has a garage
94,HasBasement,numerical,indicator,TotalBsmtSF,Indicates whether the property has a basement
95,HasFireplace,numerical,indicator,YearBuilt;YearRemodAdd,Indicates whether the property has at least on...


## Skewed feature analysis

Many machine learning algorithms perform better when numerical features have distributions that are closer to symmetric. Highly skewed variables can place disproportionate emphasis on extreme observations and may make it more difficult for some models to learn meaningful relationships.

To identify potentially problematic features, we compute the skewness of each numerical variable. Positive skewness indicates a long right tail, while negative skewness indicates a long left tail. Variables whose skewness exceeds a predefined threshold will be transformed using a logarithmic transformation.

The goal is not to force every feature into a perfectly normal distribution, but rather to reduce the influence of extreme values and produce more stable feature representations.

In [54]:
df_features["Skewness"] = np.nan

numerical_features = (
    df_features.loc[
        df_features["Type"].eq("numerical"),
        "Name"
    ]
)

for feature in numerical_features:
    df_features.loc[
        df_features["Name"].eq(feature),
        "Skewness"
    ] = skew(df_full[feature].dropna())

The skewness of each numerical feature is calculated and stored in the feature metadata table. Sorting the features by the absolute value of their skewness allows us to quickly identify variables whose distributions deviate most strongly from symmetry and are therefore candidates for logarithmic transformation.

In [55]:
numerical_non_indicator_features = df_features.loc[
    df_features["Type"].eq("numerical")
    & ~df_features["Category"].eq("indicator"),
    "Name"
].tolist()

df_features.loc[df_features["Name"].isin(numerical_non_indicator_features),
    ["Name", "Category", "Skewness"]
].sort_values(
    "Skewness",
    ascending=False
).reset_index(drop=True)

,Name,Category,Skewness
0,MiscVal,original,21.947195
1,PoolArea,original,16.898328
2,LotArea,original,12.822431
3,LowQualFinSF,original,12.088761
4,3SsnPorch,original,11.376065
5,KitchenAbvGr,original,4.302254
6,BsmtFinSF2,original,4.146143
7,EnclosedPorch,original,4.003891
8,ScreenPorch,original,3.946694
9,BsmtHalfBath,original,3.931594


Although skewness can be either positive or negative, logarithmic transformations are primarily effective at reducing positive skewness. Since the numerical features in this dataset are predominantly right-skewed, we restrict our transformations to features whose skewness exceeds a positive threshold. Features with negative skewness are left unchanged, as their distributions are generally less problematic for the models considered in this project.

There is no universally accepted skewness threshold at which a transformation becomes necessary. In practice, values between 0.5 and 1.0 are commonly used as heuristics to identify features with substantial asymmetry.

For this project, we use a threshold of **0.75**, which focuses the transformation on the most strongly right-skewed variables while avoiding unnecessary modifications to features whose distributions are only moderately skewed.

A high skewness alone, however, is not sufficient to justify a logarithmic transformation. Only **numerical, non-indicator** features are considered, as applying a logarithm to binary variables provides little practical benefit. In addition, all values of a candidate feature must be greater than **−1**, ensuring that the `log1p` transformation is mathematically well-defined and avoiding undefined values such as (\log(0)). Only features satisfying all of these conditions are selected for transformation.

In [56]:
# Conditions for log transformation
valid_cols = df_features["Name"].isin(numerical_non_indicator_features)
log_safe_mask = df_full[numerical_non_indicator_features].min() > -1
SKEW_THRESHOLD = 0.75

# Assigning LogTransform flag
df_features["LogTransform"] = False
df_features.loc[
    valid_cols &
    (df_features["Skewness"] > SKEW_THRESHOLD) &
    (df_features["Name"].map(log_safe_mask)),
    "LogTransform"
] = True

df_features

,Name,Type,Category,Source,Description,Skewness,LogTransform
0,MSSubClass,numerical,original,MSSubClass,Identifies the type of dwelling involved in th...,1.375457,True
1,MSZoning,categorical,original,MSZoning,Identifies the general zoning classification o...,NaN,False
2,LotFrontage,numerical,original,LotFrontage,Linear feet of street connected to property,0.022013,False
3,LotArea,numerical,original,LotArea,Lot size in square feet,12.822431,True
4,Street,categorical,original,Street,Type of road access to property,NaN,False
...,...,...,...,...,...,...,...
91,IsRemodeled,numerical,indicator,YearBuilt;YearRemodAdd,Indicates whether the property has been remodeled,0.138046,False
92,HasPool,numerical,indicator,PoolArea,Indicates whether the property has a pool,14.884318,False
93,HasGarage,numerical,indicator,YearBuilt;YearRemodAdd,Indicates whether the property has a garage,-3.941054,False
94,HasBasement,numerical,indicator,TotalBsmtSF,Indicates whether the property has a basement,-5.828995,False


## Encoding categorical variables

Most machine learning algorithms require numerical inputs and cannot directly process categorical variables represented as text labels. Before training our models, we therefore need to convert the categorical features into a numerical representation.

Several encoding techniques exist, including label encoding, ordinal encoding, target encoding, and one-hot encoding. For this project, we use one-hot encoding because it is simple, widely applicable, and does not impose any artificial ordering on categorical values.

One-hot encoding creates a binary feature for each category present in a variable. For example, a feature such as `Neighborhood` is transformed into a set of indicator variables, each representing the presence or absence of a specific neighborhood.

Although this process increases the number of features in the dataset, it allows machine learning models to make use of categorical information in a consistent and interpretable manner.

The generated dummy variables are not tracked individually in the `df_features` metadata table, since they are implementation artifacts derived automatically from the original categorical features rather than new conceptual features created during feature engineering.

We first identify the categorical features recorded in the metadata table and then apply one-hot encoding to the dataset.

In [57]:
categorical_features = df_features.loc[
    df_features["Type"].eq("categorical"),
    "Name"
]

df_full = pd.get_dummies(
    df_full,
    columns=categorical_features,
    dtype=int
)

## Save the processed dataset

At this stage, the dataset has undergone all planned preprocessing steps. Missing values have been handled, new features have been engineered, highly skewed numerical variables have been transformed where appropriate, and categorical features have been encoded into a numerical representation suitable for machine learning algorithms.

To avoid repeating these preprocessing steps in the next notebook, we save both the processed dataset and the updated feature metadata table. This allows the modeling stage to focus exclusively on training, evaluating, and comparing regression models.

The processed dataset now contains all information required for model development and can be loaded directly in the next notebook.

In [58]:
# Save dataframes in parquet format to preserve data types
df_full.to_parquet(
    "processed_data/02_data.parquet",
    index=False
)
df_features.to_parquet(
    "processed_data/02_features.parquet",
    index=False
)

## Summary

In this notebook, we transformed the dataset from its exploratory-analysis form into a modeling-ready representation through a combination of feature engineering and preprocessing techniques.

Key steps included:

* Creating aggregate features such as `TotalArea`, `TotalBathrooms`, and `TotalPorchArea`.
* Constructing time-based features such as `HouseAge`, `YearsSinceRemodel`, and `GarageAge`.
* Introducing interaction and ratio features such as `QualityArea`, `GarageAreaPerCar`, and `LivingAreaPerRoom`.
* Creating binary indicators describing the presence of important property characteristics.
* Combining selected amenities into the `LuxuryFeatureCount` feature.
* Documenting all engineered features through the `df_features` metadata table.
* Analyzing the skewness of numerical variables and applying logarithmic transformations to highly skewed features.
* Converting categorical variables into a numerical representation using one-hot encoding.

The resulting dataset incorporates both domain knowledge and preprocessing techniques designed to improve model performance and compatibility with machine learning algorithms. After these transformations, the dataset is fully numerical and suitable for a wide range of regression models.

To facilitate the next stage of the project, we saved both the processed dataset and the updated feature metadata table. This allows the modeling notebook to focus exclusively on training, evaluating, and comparing models rather than repeating the preprocessing pipeline.

In the [next notebook](03_modeling.ipynb), we will use this transformed dataset to train and evaluate several regression models, compare their performance, and generate a Kaggle submission.
